In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.master("local[*]").appName("spark-sql").getOrCreate()

In [3]:
spark

In [5]:
data  = spark.read.parquet("/content/yellow_tripdata_2025-11.parquet")

In [7]:
data.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [8]:
df_repartitioned = data.repartition(4)
output_path = "/content/parquet_output"
df_repartitioned.write.parquet(output_path)

In [9]:
!ls -lh /content/parquet_output

total 102M
-rw-r--r-- 1 root root 26M Mar  5 07:14 part-00000-1289ca1d-adc0-4aac-af3c-9dd03feef5d7-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  5 07:14 part-00001-1289ca1d-adc0-4aac-af3c-9dd03feef5d7-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  5 07:14 part-00002-1289ca1d-adc0-4aac-af3c-9dd03feef5d7-c000.snappy.parquet
-rw-r--r-- 1 root root 26M Mar  5 07:14 part-00003-1289ca1d-adc0-4aac-af3c-9dd03feef5d7-c000.snappy.parquet
-rw-r--r-- 1 root root   0 Mar  5 07:14 _SUCCESS


In [18]:
df_trips = spark.read.parquet('/content/parquet_output/*')

In [19]:
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|              1|          7.3|         1|                 N|         262|    

In [20]:
df_trips.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [21]:
df_trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [22]:
df_trips = df_trips.withColumn('tpep_pickup_date',F.to_date(df_trips.tpep_pickup_datetime)) \
    .withColumn('tpep_dropoff_date',F.to_date(df_trips.tpep_dropoff_datetime))

In [23]:
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|tpep_pickup_date|tpep_dropoff_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+-----------------+
|       2| 2025-11-07 15:

In [24]:
df_trips.registerTempTable('trips_data')

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [26]:
spark.sql('''
SELECT * FROM trips_data LIMIT 10
''').show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+-----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|tpep_pickup_date|tpep_dropoff_date|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+-----------------+
|       2| 2025-11-07 15:

In [28]:
spark.sql('''
SELECT COUNT(*) as Count FROM trips_data
WHERE tpep_pickup_date = '2025-11-15'
''').show()

+------+
| Count|
+------+
|162604|
+------+



In [38]:
spark.sql('''
WITH trip_duration_table AS (
    SELECT *,
    DATEDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) as trip_duration
    FROM trips_data
)
SELECT MAX(trip_duration)/60 FROM trip_duration_table LIMIT;
''').show()

+-------------------------+
|(max(trip_duration) / 60)|
+-------------------------+
|        90.63333333333334|
+-------------------------+



In [40]:
df_taxi_zone_lookup = spark.read.csv('/content/taxi_zone_lookup.csv', header=True)
df_taxi_zone_lookup.head(5)

[Row(LocationID='1', Borough='EWR', Zone='Newark Airport', service_zone='EWR'),
 Row(LocationID='2', Borough='Queens', Zone='Jamaica Bay', service_zone='Boro Zone'),
 Row(LocationID='3', Borough='Bronx', Zone='Allerton/Pelham Gardens', service_zone='Boro Zone'),
 Row(LocationID='4', Borough='Manhattan', Zone='Alphabet City', service_zone='Yellow Zone'),
 Row(LocationID='5', Borough='Staten Island', Zone='Arden Heights', service_zone='Boro Zone')]

In [41]:
df_taxi_zone_lookup.printSchema()

root
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [42]:
df_taxi_zone_lookup = df_taxi_zone_lookup.withColumn('LocationID', F.col('LocationID').cast(types.IntegerType()))
df_taxi_zone_lookup.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [43]:
df_taxi_zone_lookup.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [44]:
df_taxi_zone_lookup.registerTempTable('taxi_zone_lookup')

/usr/local/lib/python3.12/dist-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [45]:
spark.sql('''
SELECT * FROM taxi_zone_lookup
LIMIT 5
''').show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+



In [55]:
## join
spark.sql('''
WITH joined_table as (
SELECT * FROM trips_data t
JOIN taxi_zone_lookup z
ON t.PULocationID = z.LocationID)

SELECT count(*) as trip_counts,Zone FROM joined_table
group BY Zone
ORDER BY trip_counts ASC

''').show()

+-----------+--------------------+
|trip_counts|                Zone|
+-----------+--------------------+
|          1|Eltingville/Annad...|
|          1|Governor's Island...|
|          1|       Arden Heights|
|          3|       Port Richmond|
|          4|       Rikers Island|
|          4|   Rossville/Woodrow|
|          4|         Great Kills|
|          4| Green-Wood Cemetery|
|          5|         Jamaica Bay|
|         12|         Westerleigh|
|         14|New Dorp/Midland ...|
|         14|       West Brighton|
|         14|             Oakwood|
|         14|        Crotona Park|
|         15|       Willets Point|
|         16|Breezy Point/Fort...|
|         17|Saint George/New ...|
|         18|       Broad Channel|
|         21|     Mariners Harbor|
|         22|Heartland Village...|
+-----------+--------------------+
only showing top 20 rows
